Building a dataset with real speech and voice clones. Made for Google Colab.

In [ ]:
!pip -q install coqui-tts
!pip -q install librosa soundfile

import os, glob, random
import numpy as np, librosa, soundfile as sf

os.environ['COQUI_TOS_AGREED'] = '1'
SR = 16000
random.seed(0)

for d in ['real', 'fake', 'reference']:
    os.makedirs(f'/content/data/{d}', exist_ok=True)
print('folders ready')

In [ ]:
# VCTK
!pip -q install datasets
from datasets import load_dataset

SPEAKERS = ['p225','p226','p227','p228','p229','p230',
            'p231','p232','p233','p234']       # 2-3 speakers

ds = load_dataset('CSTR-Edinburgh/vctk', split='train',
                  revision='refs/convert/parquet', streaming=True)

utterances = []          # list of (speaker, text, audio_array)
counts = {s: 0 for s in SPEAKERS}
for ex in ds:
    spk = ex.get('speaker_id', '')
    if spk not in SPEAKERS or counts[spk] >= 80:
        continue
    a = ex['audio']['array'].astype(np.float32)
    fs = ex['audio']['sampling_rate']
    if fs != SR:
        a = librosa.resample(a, orig_sr=fs, target_sr=SR)
    txt = ex.get('text', '').strip()
    if len(a) < SR * 1.0 or not txt:
        continue
    utterances.append((spk, txt, a))
    counts[spk] += 1
    if all(c >= 80 for c in counts.values()):
        break

print('collected per speaker:', counts)
print('total utterances:', len(utterances))

In [ ]:
TARGET_S = 10.0
N_CLIPS_PER_SPEAKER = 12
N_REFERENCE = 5

by_speaker = {}
for spk, txt, a in utterances:
    by_speaker.setdefault(spk, []).append((txt, a))

import json, os
old = []
if os.path.exists('/content/data/manifest.json'):
    old = json.load(open('/content/data/manifest.json'))
done_speakers = {m['speaker'] for m in old}
print('already built:', done_speakers)

manifest = list(old)

for spk, items in by_speaker.items():
    if spk in done_speakers:
        print('skipping', spk); continue
    # --- reference audio: held out, never tested on ---
    ref = np.concatenate([a for _, a in items[:N_REFERENCE]])
    ref = ref / (np.max(np.abs(ref)) + 1e-9)
    sf.write(f'/content/data/reference/{spk}_ref.wav', ref[:SR*20], SR)

    # --- test clips: built from the REMAINING utterances ---
    pool = items[N_REFERENCE:]
    i, made = 0, 0
    while made < N_CLIPS_PER_SPEAKER and i < len(pool):
        chunks, texts, dur = [], [], 0.0
        while dur < TARGET_S and i < len(pool):
            txt, a = pool[i]; i += 1
            chunks.append(a); texts.append(txt)
            dur += len(a) / SR
            chunks.append(np.zeros(int(0.25 * SR)))   # small gap between sentences
            dur += 0.25
        if dur < 6.0:
            break
        y = np.concatenate(chunks)
        y = y / (np.max(np.abs(y)) + 1e-9)
        name = f'{spk}_{made:02d}.wav'
        sf.write(f'/content/data/real/{name}', y, SR)
        manifest.append({'name': name, 'speaker': spk, 'texts': texts,
                         'duration': round(len(y)/SR, 2)})
        made += 1

with open('/content/data/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=1)

print(f'built {len(manifest)} real clips')
print(f'mean duration: {np.mean([m["duration"] for m in manifest]):.1f}s')
print('reference files:', os.listdir('/content/data/reference'))

In [ ]:
import torch
import transformers.pytorch_utils as pu

# 1. Restore the function coqui-tts expects
if not hasattr(pu, 'isin_mps_friendly'):
    def isin_mps_friendly(elements, test_elements):
        return torch.isin(elements, test_elements)
    pu.isin_mps_friendly = isin_mps_friendly

_orig_load = torch.load
def _load(*a, **k):
    k['weights_only'] = False
    return _orig_load(*a, **k)
torch.load = _load

print('patches applied')

In [ ]:
import torch
from TTS.api import TTS          # import path is unchanged in the fork

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('WARNING: this will be very slow. Switch to a GPU runtime in Colab.')

tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('model loaded')

In [ ]:
def synth_sentence(text, ref_wav, out_path, max_chars=240):
    """Generate one sentence. Truncates over-long text to stay under the limit."""
    text = text.strip()
    if len(text) > max_chars:
        text = text[:max_chars].rsplit(' ', 1)[0]
    tts.tts_to_file(text=text, speaker_wav=ref_wav, language='en',
                    file_path=out_path)


os.makedirs('/content/tmp', exist_ok=True)

for k, m in enumerate(manifest):
    out = f'/content/data/fake/{m["name"]}'
    if os.path.exists(out):
        continue
    ref = f'/content/data/reference/{m["speaker"]}_ref.wav'
    chunks = []
    for j, txt in enumerate(m['texts']):
        tmp = f'/content/tmp/s{j}.wav'
        try:
            synth_sentence(txt, ref, tmp)
        except Exception as e:
            print('  synth failed:', e); continue
        a, _ = librosa.load(tmp, sr=SR)          # resample to match reals
        chunks.append(a)
        chunks.append(np.zeros(int(0.25 * SR))) # same gap as the real clips
    if not chunks:
        print('  no audio for', m['name']); continue
    y = np.concatenate(chunks)
    y = y / (np.max(np.abs(y)) + 1e-9)
    sf.write(out, y, SR)
    print(f'[{k+1}/{len(manifest)}] {m["name"]}  {len(y)/SR:.1f}s')

print('\ndone:', len(glob.glob('/content/data/fake/*.wav')), 'fake clips')

In [ ]:
def audit(folder):
    durs, rates = [], []
    for p in sorted(glob.glob(f'{folder}/*.wav')):
        info = sf.info(p)
        durs.append(info.duration); rates.append(info.samplerate)
    return np.array(durs), set(rates)

dr, rr = audit('/content/data/real')
df, rf = audit('/content/data/fake')

print(f'real: n={len(dr)}  duration {dr.mean():.1f}s +/- {dr.std():.1f}  rates={rr}')
print(f'fake: n={len(df)}  duration {df.mean():.1f}s +/- {df.std():.1f}  rates={rf}')

print('\nChecks:')
print(' sample rates match: ', rr == rf)
print(' counts within 20%:  ', abs(len(dr)-len(df)) < 0.2*max(len(dr), len(df)))
print(' durations within 2s:', abs(dr.mean()-df.mean()) < 2.0)
print('\nIf durations differ a lot, the classifier can cheat on clip length.')
print('Trim both groups to the same length if so.')

# sanity check
from IPython.display import Audio, display
print('\nreal:'); display(Audio(sorted(glob.glob('/content/data/real/*.wav'))[10]))
print('fake:'); display(Audio(sorted(glob.glob('/content/data/fake/*.wav'))[10]))